In [1]:
!git clone https://github.com/ZhangJUJU/OfficeCaltechDomainAdaptation

Cloning into 'OfficeCaltechDomainAdaptation'...
remote: Enumerating objects: 2581, done.
remote: Total 2581 (delta 0), reused 0 (delta 0), pack-reused 2581 (from 1)
Receiving objects: 100% (2581/2581), 72.20 MiB | 37.90 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Gradient Reversal Layer (usada en DANN y opcionalmente en CDAN)
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_=1.0):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)
import random
import numpy as np
import torch
from torchvision import models

from torchvision.models import resnet18, ResNet18_Weights,ResNet50_Weights,resnet50
class ResNetFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True, output_dim=512):
        super().__init__()
        base_model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(base_model.children())[:-1])  # sin la capa FC
        self.flatten = nn.Flatten()
        self.output_dim = output_dim

    def forward(self, x):
        x = self.features(x)  # [B, 512, 1, 1]
        x = self.flatten(x)   # [B, 512]
        return x


import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self, feature_dim=256, num_classes=10):
        super().__init__()
        set_seed(42)
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim=256):
        super().__init__()
        set_seed(42)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)
class DANN_ResNet(nn.Module):
    def __init__(self, num_classes=10, lambda_grl=1.0):
        super().__init__()
        set_seed(42)
        self.feature = ResNetFeatureExtractor()
        self.classifier = nn.Linear(self.feature.output_dim, num_classes)
        self.domain_discriminator = DomainDiscriminator(input_dim=self.feature.output_dim)
        self.grl = GradientReversalLayer(lambda_grl)

    def forward(self, x, mode='class'):
        f = self.feature(x)
        if mode == 'class':
            return self.classifier(f)
        elif mode == 'domain':
            return self.domain_discriminator(self.grl(f))
class ADDA_ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        set_seed(42)
        self.Fs = ResNetFeatureExtractor()
        self.Ft = ResNetFeatureExtractor()
        self.classifier = nn.Linear(self.Fs.output_dim, num_classes)
        self.domain_discriminator = DomainDiscriminator(input_dim=self.Fs.output_dim)

    def forward(self, x, domain='source', mode='class'):
        if domain == 'source':
            f = self.Fs(x)
        elif domain == 'target':
            f = self.Ft(x)
        else:
            raise ValueError("domain must be 'source' or 'target'")
        if mode == 'class':
            return self.classifier(f)
        elif mode == 'domain':
            return self.domain_discriminator(f)
        else:
            raise ValueError("mode must be 'class' or 'domain'")
class CDAN_ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        set_seed(42)
        self.feature = ResNetFeatureExtractor()
        self.classifier = nn.Linear(self.feature.output_dim, num_classes)
        self.domain_discriminator = DomainDiscriminator(input_dim=self.feature.output_dim * num_classes)

    def forward(self, x, mode='class'):
        f = self.feature(x)
        y = self.classifier(f)
        if mode == 'class':
            return y
        elif mode == 'domain':
            y_soft = F.softmax(y, dim=1)
            outer = torch.bmm(y_soft.unsqueeze(2), f.unsqueeze(1)).view(x.size(0), -1)
            return self.domain_discriminator(outer)
import torch
import torch.nn as nn
import torch.nn.functional as F

class CREDA_ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        set_seed(42)
        self.feature = ResNetFeatureExtractor()
        self.classifier = nn.Linear(self.feature.output_dim, num_classes)

    def forward(self, x, mode='class'):
        f = self.feature(x)
        y = self.classifier(f)
        if mode == 'class':
            return y  # softmax se aplica externamente si es necesario
        elif mode == 'feature':
            return f

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
import torch.serialization

torch.serialization.add_safe_globals([
    ResNetFeatureExtractor,
    Classifier,
    DomainDiscriminator,
    DANN_ResNet,
    ADDA_ResNet,
    CDAN_ResNet,
    CREDA_ResNet,
    GradientReversalFunction,
    GradientReversalLayer
])


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from itertools import permutations
from tqdm import tqdm
import numpy as np
import pandas as pd

# --- División estratificada común ---
def split_stratified(dataset, val_ratio=0.2, test_ratio=None, seed=42):
    set_seed(42)
    if hasattr(dataset, "targets"):
        labels = np.array(dataset.targets)
    elif hasattr(dataset, "labels"):
        labels = np.array(dataset.labels)
    else:
        raise ValueError("El dataset no tiene 'targets' ni 'labels'.")

    indices = np.arange(len(dataset))
    if test_ratio is None:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
        train_idx, val_idx = next(sss.split(indices, labels))
        return Subset(dataset, train_idx), Subset(dataset, val_idx)
    else:
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio + test_ratio, random_state=seed)
        train_idx, temp_idx = next(sss1.split(indices, labels))
        temp_labels = labels[temp_idx]
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio / (val_ratio + test_ratio), random_state=seed)
        val_idx, test_idx = next(sss2.split(temp_idx, temp_labels))
        val_idx = temp_idx[val_idx]
        test_idx = temp_idx[test_idx]
        return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)

# --- Batch automático ---
def util_auto_batch_size(dataset_len, max_bs=64, min_bs=16, fraction=0.1):
    bs = int(dataset_len * fraction)
    return max(min_bs, min(max_bs, bs))

# --- Evaluación común ---
import numpy as np
import torch
import torch.nn as nn
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score

def eval_model(feature_extractor, classifier, dataloader, device,num_classes):
    set_seed(42)
    feature_extractor.eval()
    classifier.eval()

    total_correct = 0
    total_samples = 0
    batch_accuracies = []
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            feats = feature_extractor(x)
            preds = classifier(feats)
            loss = loss_fn(preds, y)
            total_loss += loss.item()

            correct = (preds.argmax(1) == y).sum().item()
            batch_acc = 100.0 * correct / y.size(0)
            batch_accuracies.append(batch_acc)

            total_correct += correct
            total_samples += y.size(0)

    avg_loss = total_loss / len(dataloader)
    avg_acc = 100.0 * total_correct / total_samples
    std_acc = torch.std(torch.tensor(batch_accuracies)).item()

    return avg_loss, avg_acc, std_acc







def eval_accuracy_only(feature_extractor, classifier, dataloader, device,num_classes):
    _, acc,std = eval_model(feature_extractor, classifier, dataloader, device,num_classes)
    return acc,std
# --- Entrenamiento supervisado ---
def train_baseline(feature_extractor, classifier, train_loader, val_loader, device,num_classes, epochs=5, lr=1e-3):
    set_seed(42)
    optimizer = optim.Adam(list(feature_extractor.parameters()) + list(classifier.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        feature_extractor.train()
        classifier.train()
        total_loss, correct, total = 0, 0, 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            feats = feature_extractor(x)
            preds = classifier(feats)
            loss = criterion(preds, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (preds.argmax(1) == y).sum().item()
            total += y.size(0)
        train_acc = 100.0 * correct / total
        train_loss = total_loss / len(train_loader)
        val_loss, val_acc,_ = eval_model(feature_extractor, classifier, val_loader, device,num_classes)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.2f}% | Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.2f}%")

# --- Evaluación final común ---
def eval_accuracy_only(feature_extractor, classifier, dataloader, device,num_classes):
    _, acc,std = eval_model(feature_extractor, classifier, dataloader, device,num_classes)
    return acc,std



# --- Entrenamiento de DANN ---
# Sustituye tu versión anterior de train_dann por esta
def get_eta(epoch, total_epochs, eta_0=0.01, alpha=10, beta=0.75):
    p = epoch / total_epochs
    return eta_0 * (1 + alpha * p) ** (-beta)

def get_lambda(epoch, total_epochs, delta=10):
    p = epoch / total_epochs
    return (1 - np.exp(-delta * p)) / (1 + np.exp(-delta * p))
def train_dann(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes,epochs=5, eta_0=1e-3):
    set_seed(42)
    optimizer = optim.Adam(model.parameters(), lr=eta_0)
    criterion_class = nn.CrossEntropyLoss()
    criterion_domain = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()

        # Ajuste dinámico de tasa de aprendizaje y lambda
        new_lr = get_eta(epoch, epochs, eta_0)
        lambda_val = get_lambda(epoch, epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = new_lr

        total_cls_loss, total_dom_loss = 0, 0
        cls_correct, dom_correct, total_dom = 0, 0, 0
        tgt_iter = iter(tgt_train_loader)

        for xs, ys in tqdm(src_train_loader, desc=f"[DANN] Epoch {epoch+1}/{epochs}"):
            try:
                xt, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_train_loader)
                xt, _ = next(tgt_iter)

            # Enviar a dispositivo
            xs, ys, xt = xs.to(device), ys.to(device), xt.to(device)

            # Predicción de clase
            preds_class = model(xs, mode='class')
            loss_class = criterion_class(preds_class, ys)
            cls_correct += (preds_class.argmax(1) == ys).sum().item()

            # Discriminación de dominio
            domain_src = model(xs, mode='domain')
            domain_tgt = model(xt, mode='domain')

            bs_src = domain_src.size(0)
            bs_tgt = domain_tgt.size(0)

            domain_preds = torch.cat([domain_src, domain_tgt], dim=0)
            domain_labels = torch.cat([
                torch.ones(bs_src, 1),
                torch.zeros(bs_tgt, 1)
            ], dim=0).to(device)

            loss_domain = criterion_domain(domain_preds, domain_labels)
            dom_correct += ((domain_preds > 0.5).float() == domain_labels).sum().item()
            total_dom += domain_labels.size(0)

            # Pérdida total
            loss = loss_class + lambda_val* loss_domain
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_cls_loss += loss_class.item()
            total_dom_loss += loss_domain.item()

        val_loss_src, val_acc_src,_ = eval_model(model.feature, model.classifier, src_val_loader, device,num_classes)
        val_loss_tgt, val_acc_tgt,_ = eval_model(model.feature, model.classifier, tgt_val_loader, device,num_classes)

        print(f"Epoch {epoch+1}: LR={new_lr:.6f}, Lambda={lambda_val:.4f}")
        print(f"  Train: Cls Loss={total_cls_loss:.4f}, Dom Loss={total_dom_loss:.4f}, "
              f"Cls Acc={100. * cls_correct / len(src_train_loader.dataset):.2f}%, "
              f"Dom Acc={100. * dom_correct / total_dom:.2f}%")
        print(f"  Val Src: Loss={val_loss_src:.4f}, Acc={val_acc_src:.2f}%")
        print(f"  Val Tgt: Loss={val_loss_tgt:.4f}, Acc={val_acc_tgt:.2f}%")



# --- ADDA Entrenamiento Fase 1 ---
def train_adda_phase1(model, loader_train, loader_val, device, num_classes,epochs=5, lr=1e-3,):
    set_seed(42)
    optimizer = optim.Adam(list(model.Fs.parameters()) + list(model.classifier.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for x, y in tqdm(loader_train, desc=f"[ADDA Phase 1] Epoch {epoch+1}"):
            x, y = x.to(device), y.to(device)
            preds = model(x, domain='source', mode='class')
            loss = criterion(preds, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (preds.argmax(1) == y).sum().item()
            total += y.size(0)
        train_acc = 100.0 * correct / total
        val_loss, val_acc,_ = eval_model(model.Fs, model.classifier, loader_val, device,num_classes)
        print(f"Epoch {epoch+1}: Train Loss = {total_loss:.4f}, Train Acc = {train_acc:.2f}%, Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.2f}%")

# --- ADDA Entrenamiento Fase 2 ---
def train_adda_phase2(model, src_loader, tgt_loader, val_loader_src, val_loader_tgt, device,num_classes, epochs=5, eta_0=1e-4):
    set_seed(42)
    for p in model.Fs.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = False

    optimizer_Ft = optim.Adam(model.Ft.parameters(), lr=eta_0)
    optimizer_D = optim.Adam(model.domain_discriminator.parameters(), lr=eta_0)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()

        # Schedulers dinámicos
        new_lr = get_eta(epoch, epochs, eta_0=eta_0)
        lambda_val = get_lambda(epoch, epochs)

        for param_group in optimizer_Ft.param_groups:
            param_group['lr'] = new_lr
        for param_group in optimizer_D.param_groups:
            param_group['lr'] = new_lr

        total_domain_loss, total_target_loss = 0, 0
        dom_correct, total_dom = 0, 0
        tgt_iter = iter(tgt_loader)

        for xs, _ in tqdm(src_loader, desc=f"[ADDA Phase 2] Epoch {epoch+1}"):
            try:
                xt, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt, _ = next(tgt_iter)

            xs, xt = xs.to(device), xt.to(device)

            # Discriminador
            feats_src = model.Fs(xs).detach()
            feats_tgt = model.Ft(xt).detach()

            bs_src = feats_src.size(0)
            bs_tgt = feats_tgt.size(0)

            d_inputs = torch.cat([feats_src, feats_tgt], dim=0)
            d_labels = torch.cat([
                torch.ones(bs_src, 1),
                torch.zeros(bs_tgt, 1)
            ], dim=0).to(device)

            preds_d = model.domain_discriminator(d_inputs)
            loss_d = criterion(preds_d, d_labels)
            optimizer_D.zero_grad()
            loss_d.backward()
            optimizer_D.step()

            # Ft: Generador adversarial
            f_tgt = model.Ft(xt)
            preds_t = model.domain_discriminator(f_tgt)
            loss_t = criterion(preds_t, torch.ones_like(preds_t))
            loss_t = lambda_val * loss_t

            optimizer_Ft.zero_grad()
            loss_t.backward()
            optimizer_Ft.step()

            total_domain_loss += loss_d.item()
            total_target_loss += loss_t.item()
            dom_correct += ((preds_d > 0.5).float() == d_labels).sum().item()
            total_dom += d_labels.size(0)

        val_loss_src, val_acc_src,_ = eval_model(model.Fs, model.classifier, val_loader_src, device,num_classes)
        val_loss_tgt, val_acc_tgt,_ = eval_model(model.Ft, model.classifier, val_loader_tgt, device,num_classes)

        print(f"Epoch {epoch+1}: LR = {new_lr:.6f}, Lambda = {lambda_val:.4f}")
        print(f"  Domain Loss = {total_domain_loss:.4f}, Ft Loss = {total_target_loss:.4f}, "
              f"Domain Acc = {100. * dom_correct / total_dom:.2f}%")
        print(f"  Val Src: Loss = {val_loss_src:.4f}, Acc = {val_acc_src:.2f}%")
        print(f"  Val Tgt: Loss = {val_loss_tgt:.4f}, Acc = {val_acc_tgt:.2f}%")




# --- CDAN Entrenamiento ---
def entropy(p):
    return -torch.sum(p * torch.log(p + 1e-5), dim=1)

def train_cdan(model, src_loader, tgt_loader, val_loader_src, val_loader_tgt, device,num_classes, epochs=5, eta_0=1e-3):
    set_seed(42)
    optimizer = optim.Adam(model.parameters(), lr=eta_0)
    criterion_class = nn.CrossEntropyLoss()
    criterion_domain = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        print(f"[CDAN+E] => Epoch {epoch+1}/{epochs} - INICIO")

        model.train()
        new_lr = get_eta(epoch, epochs, eta_0)
        lambda_val = get_lambda(epoch, epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = new_lr

        total_cls_loss, total_dom_loss, total_ent_loss = 0, 0, 0
        cls_correct, dom_correct, total_dom = 0, 0, 0
        tgt_iter = iter(tgt_loader)

        for xs, ys in tqdm(src_loader, desc=f"[CDAN+E] Epoch {epoch+1}"):
            try:
                xt, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt, _ = next(tgt_iter)

            xs, ys, xt = xs.to(device), ys.to(device), xt.to(device)
            feats_src = model.feature(xs)
            preds_src = model.classifier(feats_src)
            loss_class = criterion_class(preds_src, ys)
            cls_correct += (preds_src.argmax(1) == ys).sum().item()

            feats_tgt = model.feature(xt)
            preds_tgt = model.classifier(feats_tgt)

            feats = torch.cat([feats_src, feats_tgt], dim=0)
            softmax_out = nn.Softmax(dim=1)(torch.cat([preds_src, preds_tgt], dim=0)).detach()
            joint = torch.bmm(
                softmax_out.unsqueeze(2),
                feats.unsqueeze(1)
            ).view(-1, feats.size(1) * softmax_out.size(1))

            bs_src = feats_src.size(0)
            bs_tgt = feats_tgt.size(0)
            domain_preds = model.domain_discriminator(joint)
            domain_labels = torch.cat([
                torch.ones(bs_src, 1),
                torch.zeros(bs_tgt, 1)
            ], dim=0).to(device)
            loss_domain = criterion_domain(domain_preds, domain_labels)
            dom_correct += ((domain_preds > 0.5).float() == domain_labels).sum().item()
            total_dom += domain_labels.size(0)

            # --- Entropy loss (only on target predictions) ---
            softmax_tgt = nn.Softmax(dim=1)(preds_tgt)
            loss_entropy = torch.mean(entropy(softmax_tgt))

            # --- Total loss (with entropy regularization) ---
            #(1-lambda_val)*loss_class + lambda_val * (loss_domain + loss_entropy)
            loss = 0.7*loss_class + 0.25*loss_domain + 0.05*loss_entropy

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_cls_loss += loss_class.item()
            total_dom_loss += loss_domain.item()
            total_ent_loss += loss_entropy.item()


        val_loss_src, val_acc_src,_ = eval_model(model.feature, model.classifier, val_loader_src, device,num_classes)
        val_loss_tgt, val_acc_tgt,_ = eval_model(model.feature, model.classifier, val_loader_tgt, device,num_classes)

        print(f"Epoch {epoch+1}: LR={new_lr:.6f}, Lambda={lambda_val:.4f}")
        print(f"  Cls Loss={total_cls_loss:.4f}, Dom Loss={total_dom_loss:.4f}, Ent Loss={total_ent_loss:.4f}, "
              f"Cls Acc={100. * cls_correct / len(src_loader.dataset):.2f}%, "
              f"Dom Acc={100. * dom_correct / total_dom:.2f}%")
        print(f"  Val Src: Loss={val_loss_src:.4f}, Acc={val_acc_src:.2f}%, "
              f"Val Tgt: Loss={val_loss_tgt:.4f}, Acc={val_acc_tgt:.2f}%")

# --- Ejecución CDAN ---

import torch
import torch.nn as nn
import torch.nn.functional as Fu


def renyi_entropy_order2(p):
    """p: tensor (batch_size, num_classes) con probabilidades (softmax)"""
    squared_sum = torch.sum(p ** 2, dim=1)  # suma de p_i^2 para cada muestra
    return -torch.log(squared_sum + 1e-8) / torch.log(torch.tensor(2.0, device=p.device))

class CREDALoss(nn.Module):
    def __init__(self, lambda_creda=1, sigma=1, use_entropy_weighting=True):
        super(CREDALoss, self).__init__()
        self.lambda_creda = lambda_creda
        self.sigma = sigma
        self.use_entropy_weighting = use_entropy_weighting

    def _gaussian_kernel(self, x, y):
        dist_sq = torch.cdist(x, y, p=2)**2
        return torch.exp(-dist_sq / (2 * self.sigma**2))

    def _renyi_entropy_order_2(self, K):
        if K.shape[0] == 0:
            return torch.tensor(0.0, device=K.device)
        K = K / torch.trace(K)
        return -torch.log(torch.trace(K @ K) + 1e-8) / torch.log(torch.tensor(2.0, device=K.device))

    def _mix_kernel_concat(self, K_s, K_t, K_st):
        K_mix = torch.cat([
            torch.cat([K_s, K_st], dim=1),
            torch.cat([K_st.T, K_t], dim=1)
        ], dim=0)
        return K_mix / torch.trace(K_mix)

    def forward(self, f_s, f_t, y_s, g_t):
        y_t_pseudo = torch.argmax(g_t, dim=1)
        unique_classes = torch.unique(y_s)
        total_creda_loss = 0.0
        valid_class_count = 0

        if self.use_entropy_weighting:
            entropy = -torch.sum(g_t * torch.log(g_t + 1e-8), dim=1)
            target_weights = 1.0 - entropy / torch.log(torch.tensor(g_t.shape[1], device=g_t.device, dtype=g_t.dtype))
        else:
            target_weights = None

        for c in unique_classes:
            f_s_c = f_s[y_s == c]
            f_t_c = f_t[y_t_pseudo == c]

            if f_s_c.shape[0] == 0 or f_t_c.shape[0] == 0:
                continue

            K_s_c = self._gaussian_kernel(f_s_c, f_s_c)
            K_t_c = self._gaussian_kernel(f_t_c, f_t_c)
            K_st_c = self._gaussian_kernel(f_s_c, f_t_c)

            if self.use_entropy_weighting:
                weights_c = target_weights[y_t_pseudo == c]
                W_c = torch.outer(weights_c, weights_c)
                K_t_c = K_t_c * W_c

            K_mix_c = self._mix_kernel_concat(K_s_c, K_t_c, K_st_c)

            h_s = self._renyi_entropy_order_2(K_s_c)
            h_t = self._renyi_entropy_order_2(K_t_c)
            h_mix = self._renyi_entropy_order_2(K_mix_c)

            creda_c = h_mix - 0.5 * (h_s + h_t)
            total_creda_loss += creda_c
            valid_class_count += 1

        if valid_class_count > 0:
            total_creda_loss /= valid_class_count

        return self.lambda_creda * total_creda_loss



def train_creda(model, src_loader, tgt_loader, val_loader_src, val_loader_tgt, device, num_classes, epochs=5, eta_0=1e-3):
    import torch.nn.functional as Fu
    set_seed(42)
    optimizer = optim.Adam(model.parameters(), lr=eta_0)
    criterion_class = nn.CrossEntropyLoss()
    creda_loss_fn = CREDALoss(lambda_creda=1, sigma=1.0,use_entropy_weighting=True)

    for epoch in range(epochs):
        print(f"[CREDA] => Epoch {epoch+1}/{epochs} - INICIO")

        model.train()
        new_lr = get_eta(epoch, epochs, eta_0)
        lambda_val = get_lambda(epoch, epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = new_lr

        total_cls_loss, total_creda_loss = 0, 0
        cls_correct = 0
        tgt_iter = iter(tgt_loader)

        for xs, ys in tqdm(src_loader, desc=f"[CREDA] Epoch {epoch+1}"):
            try:
                xt, _ = next(tgt_iter)
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xt, _ = next(tgt_iter)

            xs, ys, xt = xs.to(device), ys.to(device), xt.to(device)

            feats_src = model.forward(xs, mode='feature')
            preds_src = model.forward(xs, mode='class')
            loss_class = criterion_class(preds_src, ys)
            cls_correct += (preds_src.argmax(1) == ys).sum().item()

            feats_tgt = model.forward(xt, mode='feature')
            preds_tgt = model.forward(xt, mode='class')
            softmax_tgt = Fu.softmax(preds_tgt, dim=1)
            loss_creda = creda_loss_fn(feats_src, feats_tgt, ys, softmax_tgt)
            loss_entropy = torch.mean(renyi_entropy_order2(softmax_tgt))

            #loss = 0.8* loss_class + 0.05 * loss_creda + 0.15 * loss_entropy
            loss = loss_class + (0.05)*(loss_creda +  loss_entropy)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_cls_loss += loss_class.item()
            total_creda_loss += loss_creda.item()

        val_loss_src, val_acc_src, _ = eval_model(model.feature, model.classifier, val_loader_src, device, num_classes)
        val_loss_tgt, val_acc_tgt, _ = eval_model(model.feature, model.classifier, val_loader_tgt, device, num_classes)

        print(f"Epoch {epoch+1}: LR={new_lr:.6f}, Lambda={lambda_val:.4f}")
        print(f"  Cls Loss={total_cls_loss:.4f}, CREDA Loss={total_creda_loss:.4f}, "
              f"Cls Acc={100. * cls_correct / len(src_loader.dataset):.2f}%")
        print(f"  Val Src: Loss={val_loss_src:.4f}, Acc={val_acc_src:.2f}%, "
              f"Val Tgt: Loss={val_loss_tgt:.4f}, Acc={val_acc_tgt:.2f}%")

def run_baseline(dataset_key, sets_dict, epochs=5, val_split=0.2, save=False,specific_pair=None):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    models = {}
    pairs = [specific_pair] if specific_pair else permutations(sets_dict.keys(), 2)

    for src, tgt in pairs:
        print(f"\n[{dataset_key}] Baseline {src} → {tgt}")
        src_data = sets_dict[src]
        tgt_data = sets_dict[tgt]
        targets = src_data.targets if hasattr(src_data, 'targets') else src_data.labels
        num_classes = len(set(targets.tolist() if hasattr(targets, 'tolist') else list(targets)))

        src_train, src_val = split_stratified(src_data, val_ratio=val_split)
        train_loader = DataLoader(src_train, batch_size=util_auto_batch_size(len(src_train)), shuffle=True)
        val_loader = DataLoader(src_val, batch_size=256, shuffle=False)
        tgt_loader = DataLoader(tgt_data, batch_size=8, shuffle=False)

        F = ResNetFeatureExtractor(pretrained=True).to(device)
        C = Classifier(feature_dim=512, num_classes=num_classes).to(device)
        train_baseline(F, C, train_loader, val_loader, device, epochs=epochs, num_classes=num_classes)
        acc, std = eval_accuracy_only(F, C, tgt_loader, device, num_classes)

        results.append({
            "Dataset": dataset_key+'Baseline',
            "Source": src,
            "Target": tgt,
            "Test Accuracy": acc,
            'std': std
        })

        if save:
            models[(src, tgt)] = (F, C)

    return (pd.DataFrame(results), models) if save else pd.DataFrame(results)
def run_dann(dataset_key, sets_dict, epochs=5, save=False,specific_pair=None):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    models = {}
    pairs = [specific_pair] if specific_pair else permutations(sets_dict.keys(), 2)

    for src, tgt in pairs:
        print(f"\n[{dataset_key}] DANN {src} → {tgt}")
        src_data = sets_dict[src]
        tgt_data = sets_dict[tgt]
        src_train, src_val, src_test = split_stratified(src_data, val_ratio=0.15, test_ratio=0.15)
        tgt_train, tgt_val, tgt_test = split_stratified(tgt_data, val_ratio=0.15, test_ratio=0.15)

        targets = src_data.targets if hasattr(src_data, 'targets') else src_data.labels
        num_classes = len(set(targets.tolist() if hasattr(targets, 'tolist') else list(targets)))

        src_train_loader = DataLoader(src_train, batch_size=util_auto_batch_size(len(src_train)), shuffle=True)
        tgt_train_loader = DataLoader(tgt_train, batch_size=util_auto_batch_size(len(tgt_train)), shuffle=True)
        src_val_loader = DataLoader(src_val, batch_size=256, shuffle=False)
        tgt_val_loader = DataLoader(tgt_val, batch_size=256, shuffle=False)
        tgt_test_loader = DataLoader(tgt_test, batch_size=8, shuffle=False)

        model = DANN_ResNet(num_classes=num_classes, lambda_grl=1.0).to(device)
        train_dann(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, epochs=epochs, num_classes=num_classes)
        acc, std = eval_accuracy_only(model.feature, model.classifier, tgt_test_loader, device, num_classes)

        results.append({
            "Dataset": dataset_key+'DANN',
            "Source": src,
            "Target": tgt,
            "Test Accuracy": acc,
            'std': std
        })

        if save:
            models[(src, tgt)] = model

    return (pd.DataFrame(results), models) if save else pd.DataFrame(results)
def run_adda(dataset_key, sets_dict, epochs_cl=5, epochs_dc=5, save=False,specific_pair=None):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    pretrained_sources = {}
    models = {}
    pairs = [specific_pair] if specific_pair else permutations(sets_dict.keys(), 2)

    for src, tgt in pairs:
        print(f"\n[{dataset_key}] ADDA {src} → {tgt}")
        src_data = sets_dict[src]
        tgt_data = sets_dict[tgt]
        src_train, src_val, src_test = split_stratified(src_data, val_ratio=0.15, test_ratio=0.15)
        tgt_train, tgt_val, tgt_test = split_stratified(tgt_data, val_ratio=0.15, test_ratio=0.15)

        targets = src_data.targets if hasattr(src_data, 'targets') else src_data.labels
        num_classes = len(set(targets.tolist() if hasattr(targets, 'tolist') else list(targets)))

        src_train_loader = DataLoader(src_train, batch_size=util_auto_batch_size(len(src_train)), shuffle=True)
        tgt_train_loader = DataLoader(tgt_train, batch_size=util_auto_batch_size(len(tgt_train)), shuffle=True)
        src_val_loader = DataLoader(src_val, batch_size=256, shuffle=False)
        tgt_val_loader = DataLoader(tgt_val, batch_size=256, shuffle=False)
        tgt_test_loader = DataLoader(tgt_test, batch_size=8, shuffle=False)

        model = ADDA_ResNet(num_classes=num_classes).to(device)

        if src not in pretrained_sources:
            print("\nPhase 1: Entrenamiento supervisado en dominio fuente")
            train_adda_phase1(model, src_train_loader, src_val_loader, device, epochs=epochs_cl, num_classes=num_classes)
            pretrained_sources[src] = {"Fs": model.Fs.state_dict(), "C": model.classifier.state_dict()}
        else:
            print(f"\nPhase 1: Reutilizando pesos entrenados para {src}")
            model.Fs.load_state_dict(pretrained_sources[src]["Fs"])
            model.classifier.load_state_dict(pretrained_sources[src]["C"])

        print("\nPhase 2: Adaptación adversarial con Ft")
        model.Ft.load_state_dict(model.Fs.state_dict())
        train_adda_phase2(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, epochs=epochs_dc, num_classes=num_classes)

        test_loss, test_acc, std = eval_model(model.Ft, model.classifier, tgt_test_loader, device, num_classes)

        results.append({
            "Dataset": dataset_key+'ADDA',
            "Source": src,
            "Target": tgt,
            "Test Accuracy": test_acc,
            'std': std
        })

        if save:
            models[(src, tgt)] = model

    return (pd.DataFrame(results), models) if save else pd.DataFrame(results)
def run_cdan(dataset_key, sets_dict, epochs=5, save=False,specific_pair=None):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    models = {}
    pairs = [specific_pair] if specific_pair else permutations(sets_dict.keys(), 2)

    for src, tgt in pairs:
        print(f"\n[{dataset_key}] CDAN {src} → {tgt}")
        src_data = sets_dict[src]
        tgt_data = sets_dict[tgt]
        src_train, src_val, src_test = split_stratified(src_data, val_ratio=0.15, test_ratio=0.15)
        tgt_train, tgt_val, tgt_test = split_stratified(tgt_data, val_ratio=0.15, test_ratio=0.15)

        targets = src_data.targets if hasattr(src_data, 'targets') else src_data.labels
        num_classes = len(set(targets.tolist() if hasattr(targets, 'tolist') else list(targets)))

        src_train_loader = DataLoader(src_train, batch_size=util_auto_batch_size(len(src_train)), shuffle=True)
        tgt_train_loader = DataLoader(tgt_train, batch_size=util_auto_batch_size(len(tgt_train)), shuffle=True)
        src_val_loader = DataLoader(src_val, batch_size=256, shuffle=False)
        tgt_val_loader = DataLoader(tgt_val, batch_size=256, shuffle=False)
        tgt_test_loader = DataLoader(tgt_test, batch_size=8, shuffle=False)

        model = CDAN_ResNet(num_classes=num_classes).to(device)
        train_cdan(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, epochs=epochs, num_classes=num_classes)
        test_loss, test_acc, std = eval_model(model.feature, model.classifier, tgt_test_loader, device, num_classes)

        results.append({
            "Dataset": dataset_key+'CDAN',
            "Source": src,
            "Target": tgt,
            "Test Accuracy": test_acc,
            'std': std
        })

        if save:
            models[(src, tgt)] = model

    return (pd.DataFrame(results), models) if save else pd.DataFrame(results)
def run_creda(dataset_key, sets_dict, epochs=5, save=False,specific_pair=None):
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []
    models = {}
    pairs = [specific_pair] if specific_pair else permutations(sets_dict.keys(), 2)

    for src, tgt in pairs:
        print(f"\n[{dataset_key}] CREDA {src} → {tgt}")
        src_data = sets_dict[src]
        tgt_data = sets_dict[tgt]
        src_train, src_val, src_test = split_stratified(src_data, val_ratio=0.15, test_ratio=0.15)
        tgt_train, tgt_val, tgt_test = split_stratified(tgt_data, val_ratio=0.15, test_ratio=0.15)

        targets = src_data.targets if hasattr(src_data, 'targets') else src_data.labels
        num_classes = len(set(targets.tolist() if hasattr(targets, 'tolist') else list(targets)))

        src_train_loader = DataLoader(src_train, batch_size=util_auto_batch_size(len(src_train)), shuffle=True)
        tgt_train_loader = DataLoader(tgt_train, batch_size=util_auto_batch_size(len(tgt_train)), shuffle=True)
        src_val_loader = DataLoader(src_val, batch_size=256, shuffle=False)
        tgt_val_loader = DataLoader(tgt_val, batch_size=256, shuffle=False)
        tgt_test_loader = DataLoader(tgt_test, batch_size=8, shuffle=False)

        model = CREDA_ResNet(num_classes=num_classes).to(device)
        if src == 'M' and tgt == 'S':
            train_creda(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes, epochs=1)
        elif src == 'S' and tgt == 'M':
            train_creda(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes, epochs=5)
        elif src == 'U' and tgt == 'S':
            train_creda(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes, epochs=3)
        elif src == 'S' and tgt == 'U':
            train_creda(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes, epochs=7)
        
        else:
            train_creda(model, src_train_loader, tgt_train_loader, src_val_loader, tgt_val_loader, device, num_classes, epochs=epochs)
        test_loss, test_acc, std = eval_model(model.feature, model.classifier, tgt_test_loader, device, num_classes)

        results.append({
            "Dataset": dataset_key+'CREDA',
            "Source": src,
            "Target": tgt,
            "Test Accuracy": test_acc,
            "std": std
        })

        if save:
            models[(src, tgt)] = model

    return (pd.DataFrame(results), models) if save else pd.DataFrame(results)
import os
import torch

def run_all_models(combinations_dict, sets_all, output_dir="saved_models", epochs=5):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for dataset_key, (src, tgt) in combinations_dict.items():
        print(f"\n📦 Ejecutando modelos para: {dataset_key} → {src} → {tgt}")
        sets_dict = sets_all[dataset_key]
        subset = {src: sets_dict[src], tgt: sets_dict[tgt]}

        # 1. BASELINE
        df_baseline, models_baseline = run_baseline(dataset_key, subset, epochs=epochs, save=True, specific_pair=(src, tgt))
        all_results.append(df_baseline)
        for (s, t), (F, C) in models_baseline.items():
            torch.save(F.state_dict(), os.path.join(output_dir, f"Baseline_{s}_{t}_F.pth"))
            torch.save(C.state_dict(), os.path.join(output_dir, f"Baseline_{s}_{t}_C.pth"))

        # 2. DANN
        df_dann, models_dann = run_dann(dataset_key, subset, epochs=epochs, save=True, specific_pair=(src, tgt))
        all_results.append(df_dann)
        for (s, t), model in models_dann.items():
            torch.save(model.state_dict(), os.path.join(output_dir, f"DANN_{s}_{t}_weights.pth"))

        # 3. ADDA
        df_adda, models_adda = run_adda(dataset_key, subset, epochs_cl=epochs, epochs_dc=epochs, save=True, specific_pair=(src, tgt))
        all_results.append(df_adda)
        for (s, t), model in models_adda.items():
            torch.save(model.state_dict(), os.path.join(output_dir, f"ADDA_{s}_{t}_weights.pth"))

        # 4. CDAN
        df_cdan, models_cdan = run_cdan(dataset_key, subset, epochs=epochs, save=True, specific_pair=(src, tgt))
        all_results.append(df_cdan)
        for (s, t), model in models_cdan.items():
            torch.save(model.state_dict(), os.path.join(output_dir, f"CDAN_{s}_{t}_weights.pth"))

        # 5. CREDA
        if dataset_key == 'MNIST-USPS-SVHN':
            df_creda, models_creda = run_creda(dataset_key, subset, epochs=7, save=True, specific_pair=(src, tgt))
        else:
            df_creda, models_creda = run_creda(dataset_key, subset, epochs=epochs, save=True, specific_pair=(src, tgt))
        
        all_results.append(df_creda)
        for (s, t), model in models_creda.items():
            torch.save(model.state_dict(), os.path.join(output_dir, f"CREDA_{s}_{t}_weights.pth"))

    return pd.concat(all_results, ignore_index=True)


In [4]:
!pip install -q gdown

# Descargar directamente desde el enlace de Google Drive (ID del archivo)
!gdown --id 1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf -O ImageCLEF-DA.zip

import zipfile
import os

zip_path = "ImageCLEF-DA.zip"
extract_dir = "ImageCLEF-DA"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Verificar contenido
os.listdir(extract_dir)


/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf
From (redirected): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf&confirm=t&uuid=0f3c7784-4c07-4ea8-8395-1ce34d55a71b
To: /kaggle/working/ImageCLEF-DA.zip
100%|█████████████████████████████████████████| 225M/225M [00:01<00:00, 126MB/s]


['image_CLEF']

In [5]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

set_seed(42)
# Transforms
transform_digits = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

transform_clef = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset loading
mnist_full = datasets.MNIST(root='.', train=True, download=True, transform=transform_digits)
usps_full = datasets.USPS(root='.', train=True, download=True, transform=transform_digits)
svhn_full = datasets.SVHN(root='.', split='train', download=True, transform=transform_digits)

I_full = ImageFolder("/kaggle/working/ImageCLEF-DA/image_CLEF/i", transform=transform_clef)
P_full = ImageFolder("/kaggle/working/ImageCLEF-DA/image_CLEF/p", transform=transform_clef)
C_full = ImageFolder("/kaggle/working/ImageCLEF-DA/image_CLEF/c", transform=transform_clef)

A_full = ImageFolder("/kaggle/working/OfficeCaltechDomainAdaptation/images/amazon", transform=transform_clef)
W_full = ImageFolder("/kaggle/working/OfficeCaltechDomainAdaptation/images/webcam", transform=transform_clef)
D_full = ImageFolder("/kaggle/working/OfficeCaltechDomainAdaptation/images/dslr", transform=transform_clef)

# Agrupación
set_1 = {"M": mnist_full, "U": usps_full, "S": svhn_full}
set_2 = {"I": I_full, "P": P_full, "C": C_full}
set_3 = {"A": A_full, "W": W_full, "D": D_full}
sets_all = {
    "MNIST-USPS-SVHN": set_1,
    "ImageCLEF": set_2,
    "Office-Caltech": set_3
}
combinations_dict = {
    "MNIST-USPS-SVHN": ("M", "U") # Webcam → DSLR
}


100%|██████████| 9.91M/9.91M [00:00<00:00, 31.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 770kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 6.94MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.81MB/s]
100%|██████████| 6.58M/6.58M [00:01<00:00, 4.51MB/s]
100%|██████████| 182M/182M [00:09<00:00, 19.6MB/s]


In [6]:
results_df = run_all_models(combinations_dict, sets_all, output_dir="modelos_entrenados", epochs=20)
results_df.to_csv("resultados_adaptacion.csv", index=False)


📦 Ejecutando modelos para: MNIST-USPS-SVHN → M → U

[MNIST-USPS-SVHN] Baseline M → U


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 138MB/s]
Epoch 1/20: 100%|██████████| 750/750 [00:30<00:00, 24.69it/s]


Epoch 1: Train Loss = 0.1746, Train Acc = 95.82% | Val Loss = 0.1194, Val Acc = 96.47%


Epoch 2/20: 100%|██████████| 750/750 [00:28<00:00, 26.08it/s]


Epoch 2: Train Loss = 0.0729, Train Acc = 98.15% | Val Loss = 0.0481, Val Acc = 98.79%


...

Epoch 5: LR=0.000240, Lambda=0.9934
  Cls Loss=6.8466, CREDA Loss=592.5290, Cls Acc=99.70%
  Val Src: Loss=0.0807, Acc=98.36%, Val Tgt: Loss=0.1085, Acc=97.71%
[CREDA] => Epoch 6/7 - INICIO


[CREDA] Epoch 6: 100%|██████████| 657/657 [01:31<00:00,  7.19it/s]


Epoch 6: LR=0.000207, Lambda=0.9984
  Cls Loss=6.5565, CREDA Loss=588.6888, Cls Acc=99.72%
  Val Src: Loss=0.0485, Acc=99.03%, Val Tgt: Loss=0.1045, Acc=97.53%
[CREDA] => Epoch 7/7 - INICIO


[CREDA] Epoch 7: 100%|██████████| 657/657 [01:32<00:00,  7.10it/s]


Epoch 7: LR=0.000184, Lambda=0.9996
  Cls Loss=5.6906, CREDA Loss=584.2508, Cls Acc=99.77%
  Val Src: Loss=0.0606, Acc=98.68%, Val Tgt: Loss=0.1131, Acc=97.26%


In [7]:
results_df

,Dataset,Source,Target,Test Accuracy,std
0,MNIST-USPS-SVHNBaseline,M,U,24.194212,15.659407
1,MNIST-USPS-SVHNDANN,M,U,87.202925,12.264993
2,MNIST-USPS-SVHNADDA,M,U,94.332724,7.105236
3,MNIST-USPS-SVHNCDAN,M,U,88.574040,10.333863
4,MNIST-USPS-SVHNCREDA,M,U,97.166362,5.463932


In [8]:
import torch

def load_model(model_type, src, tgt, path="saved_models", num_classes=10):
    model_type = model_type.lower()
    
    if model_type == "baseline":
        F = ResNetFeatureExtractor()
        C = Classifier(feature_dim=512, num_classes=num_classes)
        F.load_state_dict(torch.load(f"{path}/Baseline_{src}_{tgt}_F.pth"))
        C.load_state_dict(torch.load(f"{path}/Baseline_{src}_{tgt}_C.pth"))
        F.eval()
        C.eval()
        return F, C

    elif model_type == "dann":
        model = DANN_ResNet(num_classes=num_classes)
        model.load_state_dict(torch.load(f"{path}/DANN_{src}_{tgt}_weights.pth"))
        model.eval()
        return model

    elif model_type == "adda":
        model = ADDA_ResNet(num_classes=num_classes)
        model.load_state_dict(torch.load(f"{path}/ADDA_{src}_{tgt}_weights.pth"))
        model.eval()
        return model

    elif model_type == "cdan":
        model = CDAN_ResNet(num_classes=num_classes)
        model.load_state_dict(torch.load(f"{path}/CDAN_{src}_{tgt}_weights.pth"))
        model.eval()
        return model

    elif model_type == "creda":
        model = CREDA_ResNet(num_classes=num_classes)
        model.load_state_dict(torch.load(f"{path}/CREDA_{src}_{tgt}_weights.pth"))
        model.eval()
        return model

    else:
        raise ValueError(f"Modelo no reconocido: {model_type}")
def extract_features(model, dataloader, device="cuda"):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            feats = model.feature(x)
            features.append(feats.cpu())
            labels.append(y)

    features = torch.cat(features, dim=0)
    labels = torch.cat(labels, dim=0)
    return features, labels
def extract_features_baseline(F, dataloader, device="cuda"):
    F.eval()
    features = []
    labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            feats = F(x)
            features.append(feats.cpu())
            labels.append(y)

    return torch.cat(features), torch.cat(labels)


In [9]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# sets_dict = sets_all["MNIST-USPS-SVHN"]
# src, tgt = combinations_dict["MNIST-USPS-SVHN"]
# src_data = sets_dict[src]
# tgt_data = sets_dict[tgt]

# # División
# src_train, src_val, src_test = split_stratified(src_data, val_ratio=0.15, test_ratio=0.15)
# tgt_train, tgt_val, tgt_test = split_stratified(tgt_data, val_ratio=0.15, test_ratio=0.15)

# # Loaders
# src_test_loader = DataLoader(src_test, batch_size=64, shuffle=False)
# tgt_test_loader = DataLoader(tgt_test, batch_size=64, shuffle=False)
# for x,y in src_test_loader:
#     print(x.shape,y.shape)
# # Cargar modelo
# #model = load_model("creda", src, tgt, path="modelos_entrenados", num_classes=10).to(device)

# # Extraer representaciones
# #features_src, labels_src = extract_features(model, src_test_loader, device)
# #features_tgt, labels_tgt = extract_features(model, tgt_test_loader, device)


#Baseline

In [10]:
# # # # --- Entrenamiento Baseline para cada set ---
# results_baseline_1 = run_baseline(dataset_key="Digits", sets_dict=set_1, epochs=20)
# results_baseline_1.to_csv("Digits_baseline_1.csv", index=False)
# # results_baseline_2 = run_baseline(dataset_key="ImageCLEF", sets_dict=set_2, epochs=20)
# # results_baseline_2.to_csv("ImageCLEF_baseline_1.csv", index=False)
# # results_baseline_3 = run_baseline(dataset_key="Office-Caltech", sets_dict=set_3, epochs=20)
# # results_baseline_3.to_csv("Office-Caltech_baseline_1.csv", index=False)


In [11]:
# results_baseline_1

#DANN

In [12]:
# # # # --- Entrenamiento DANN para cada set ---
# results_dann_1 = run_dann(dataset_key="Digits", sets_dict=set_1, epochs=20)
# results_dann_1.to_csv("Digits_dann_1.csv", index=False)
# # results_dann_2 = run_dann(dataset_key="ImageCLEF", sets_dict=set_2, epochs=20)
# # results_dann_2.to_csv("ImageCLEF_dann_1.csv", index=False)
# # results_dann_3 = run_dann(dataset_key="Office-Caltech", sets_dict=set_3, epochs=20)
# # results_dann_3.to_csv("Office-Caltech_dann_1.csv", index=False)


In [13]:
# results_dann_1

#ADDA

In [14]:
# # # --- Entrenamiento ADDA para cada set (clasificador + adaptador) ---
# results_adda_1 = run_adda(dataset_key="Digits", sets_dict=set_1, epochs_cl=20, epochs_dc=20)
# results_adda_1.to_csv("Digits_adda_1.csv", index=False)
# # results_adda_2 = run_adda(dataset_key="ImageCLEF", sets_dict=set_2, epochs_cl=20, epochs_dc=20)
# # results_adda_2.to_csv("ImageCLEF_adda_1.csv", index=False)
# # results_adda_3 = run_adda(dataset_key="Office-Caltech", sets_dict=set_3, epochs_cl=20, epochs_dc=20)
# # results_adda_3.to_csv("Office-Caltech_adda_1.csv", index=False)

In [15]:
# results_adda_1

# CDAN

In [16]:
# results_cdan_1 = run_cdan(dataset_key="Digits", sets_dict=set_1, epochs=20)
# results_cdan_1.to_csv("Digits_cdan_1.csv", index=False)
# # results_cdan_2 = run_cdan(dataset_key="ImageCLEF", sets_dict=set_2, epochs=20)
# # results_cdan_2.to_csv("ImageCLEF_cdan_1.csv", index=False)
# # results_cdan_3 = run_cdan(dataset_key="Office-Caltech", sets_dict=set_3, epochs=20)
# # results_cdan_3.to_csv("Office-Caltech_cdan_1.csv", index=False)

In [17]:
# results_cdan_1

In [18]:
# results_creda_3 = run_creda(dataset_key="Office-Caltech", sets_dict=set_3, epochs=20)
# results_creda_3.to_csv("Office-Caltech_cdan_1.csv", index=False)

In [19]:
# results_creda_3['Test Accuracy'].mean(),results_creda_3['Test Accuracy'].std()

In [20]:
# results_creda_3

In [21]:
# results_creda_2 = run_creda(dataset_key="ImageCLEF", sets_dict=set_2, epochs=20)
# results_creda_2.to_csv("ImageCLEF_cdan_1.csv", index=False)

In [22]:
# results_creda_2['Test Accuracy'].mean(),results_creda_2['Test Accuracy'].std()

In [23]:
# results_creda_2

In [24]:
# results_creda_1 = run_creda(dataset_key="Digits", sets_dict=set_1, epochs=5)
# results_creda_1.to_csv("Digits_cdan_1.csv", index=False)

In [25]:
# results_creda_1['Test Accuracy'].mean(),results_creda_1['std'].mean()

In [26]:
# results_creda_1